# 01. 데이터 수집 및 탐색적 분석 (EDA)

## 연구 목적
USGBC에서 수집한 한국 LEED 인증 건물 데이터를 로딩하고,
버전별 분포, 등급 분포, 카테고리 점수 현황을 파악합니다.

## 데이터 출처
- USGBC 프로젝트 디렉토리: https://www.usgbc.org/projects?Country=%5B%22Republic+of+Korea%22%5D
- LEED Scorecard PDF: 개별 프로젝트 성적표

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
from pathlib import Path

matplotlib.rcParams['font.family'] = 'Malgun Gothic'
matplotlib.rcParams['axes.unicode_minus'] = False

from src.data.loader import LEEDDataLoader

print('라이브러리 로딩 완료')

## 1. 데이터 로딩

### [데이터 파일 안내]
실제 연구를 위해 아래 두 가지 데이터가 필요합니다:

1. **`data/raw/PublicLEEDProjectDirectory.xlsx`**
   - 다운로드: https://www.usgbc.org/projects?Country=%5B%22Republic+of+Korea%22%5D
   - 우측 상단 'Export' 버튼 클릭 → Excel 다운로드
   - `data/raw/` 폴더에 저장

2. **`data/raw/scorecards/` 폴더**
   - 각 프로젝트의 Scorecard PDF를 다운로드
   - USGBC 프로젝트 페이지 → 'Scorecard' 탭 → PDF 다운로드
   - `data/raw/scorecards/` 폴더에 저장

실제 데이터가 없으면 아래 샘플 데이터로 실습 가능합니다.

In [ ]:
loader = LEEDDataLoader(data_dir='../data/raw')

# ─────────────────────────────────────────────────────────────────────
# 실제 데이터 파일이 있을 경우:
# ─────────────────────────────────────────────────────────────────────
# df = loader.load_korea_projects('PublicLEEDProjectDirectory.xlsx')

# ─────────────────────────────────────────────────────────────────────
# 데이터가 없을 경우 → 샘플 데이터 사용
# ─────────────────────────────────────────────────────────────────────
df = loader.create_sample_data()

print(f'\n데이터 크기: {df.shape}')
df.head()

In [ ]:
# 기본 통계 정보
print('=== 데이터 기본 정보 ===')
print(df.dtypes)
print('\n=== 결측치 현황 ===')
print(df.isnull().sum())

## 2. 버전별 분포 분석

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 버전별 분포
version_counts = df['version'].value_counts()
axes[0].pie(version_counts.values, labels=version_counts.index,
            autopct='%1.1f%%', startangle=90,
            colors=['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4'])
axes[0].set_title('LEED 버전별 분포\n(한국 인증 건물)', fontsize=13)

# 등급별 분포
grade_order = ['Certified', 'Silver', 'Gold', 'Platinum']
grade_counts = df['certification_level'].value_counts().reindex(grade_order, fill_value=0)
colors = ['#e8e8e8', '#C0C0C0', '#FFD700', '#E5E4E2']
bars = axes[1].bar(grade_counts.index, grade_counts.values, color=colors, edgecolor='gray')
axes[1].set_title('LEED 등급별 분포\n(한국 인증 건물)', fontsize=13)
axes[1].set_ylabel('건물 수')
for bar, val in zip(bars, grade_counts.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 str(val), ha='center', fontsize=11)

plt.tight_layout()
plt.savefig('../outputs/figures/01_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. 카테고리별 점수 분포 분석

In [ ]:
score_cols = [c for c in df.columns if c.startswith('score_') and not c.startswith('score_v5_')]
print(f'점수 컬럼: {score_cols}')

# 카테고리별 달성률 계산
from src.data.loader import LEED_VERSION_MAX_SCORES

achievement_records = []
for _, row in df.iterrows():
    ver = row.get('version', 'v4')
    max_scores = LEED_VERSION_MAX_SCORES.get(ver, LEED_VERSION_MAX_SCORES['v4'])
    for cat in ['SS', 'WE', 'EA', 'MR', 'IEQ', 'IN']:
        raw = row.get(f'score_{cat}', 0) or 0
        max_s = max_scores.get(cat, 1)
        achievement_records.append({
            'category': cat,
            'achievement_rate': (raw / max_s * 100) if max_s > 0 else 0,
            'grade': row.get('certification_level', 'Unknown'),
        })

achievement_df = pd.DataFrame(achievement_records)

fig, ax = plt.subplots(figsize=(12, 6))
grade_order = ['Certified', 'Silver', 'Gold', 'Platinum']
available_grades = [g for g in grade_order if g in achievement_df['grade'].unique()]

achievement_pivot = achievement_df.groupby(['category', 'grade'])['achievement_rate'].mean().unstack(fill_value=0)
achievement_pivot = achievement_pivot.reindex(columns=available_grades, fill_value=0)

achievement_pivot.plot(kind='bar', ax=ax,
                       color=['#e8e8e8', '#C0C0C0', '#FFD700', '#E5E4E2'][:len(available_grades)],
                       edgecolor='gray')
ax.set_title('등급별 카테고리 달성률 비교', fontsize=13)
ax.set_xlabel('카테고리')
ax.set_ylabel('달성률 (%)')
ax.legend(title='등급', loc='upper right')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
plt.tight_layout()
plt.savefig('../outputs/figures/01_category_achievement.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. 버전별 카테고리 점수 구조 차이 시각화

LangGraph 표준화가 필요한 이유를 보여주는 핵심 분석

In [ ]:
from src.data.loader import LEED_VERSION_MAX_SCORES

versions = ['v2.2', 'v3', 'v4', 'v5']
categories = ['SS', 'WE', 'EA', 'MR', 'IEQ', 'IN', 'RP', 'LT', 'IP']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for idx, ver in enumerate(versions):
    max_scores = LEED_VERSION_MAX_SCORES[ver]
    cats = [c for c in categories if c in max_scores]
    vals = [max_scores[c] for c in cats]
    total = max_scores.get('TOTAL', sum(vals))

    wedges, texts, autotexts = axes[idx].pie(
        vals, labels=cats, autopct='%1.1f%%',
        startangle=90,
    )
    axes[idx].set_title(f'LEED {ver}\n(총점: {total}점)', fontsize=12)

plt.suptitle('LEED 버전별 카테고리 배점 구조 비교\n→ 직접 비교 불가, 표준화 필요!',
             fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('../outputs/figures/01_version_structure.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n버전별 최대 배점 비교:')
rows = []
for ver in versions:
    row = {'Version': ver}
    for cat in categories:
        row[cat] = LEED_VERSION_MAX_SCORES[ver].get(cat, '-')
    row['Total'] = LEED_VERSION_MAX_SCORES[ver].get('TOTAL', '-')
    rows.append(row)
pd.DataFrame(rows)

## 5. 데이터 저장

In [ ]:
# ─────────────────────────────────────────────────────────────────────
# 실제 데이터를 로딩했을 경우, 전처리 전 원본 데이터를 저장
# ─────────────────────────────────────────────────────────────────────
save_path = '../data/processed/korea_leed_raw.csv'
df.to_csv(save_path, index=False, encoding='utf-8-sig')
print(f'데이터 저장 완료: {save_path}')
print(f'   총 {len(df)}개 프로젝트')